# 03 — Ajuste do Pipeline de Treinamento
**Objetivo:** Melhorar o desempenho do modelo baseline aplicando ajustes no pré-processamento e na vetorização.

### Ajustes realizados vs Baseline

| Componente | Baseline | Ajustado |
|---|---|---|
| Filtro mín. reviews/produto | 5 | 10 |
| Filtro mín. reviews/usuário | — | 5 |
| Limpeza de texto | stopwords sklearn | + stemming + remoção HTML/URL |
| TF-IDF features | 8.000 | 15.000 |
| N-gramas | (1,1) | (1,2) |
| Sublinear TF | Não | Sim |
| Ponderação perfil usuário | Média simples | Ponderada pelo rating |


In [ ]:
import json, re, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Stemmer (sem depender de download de rede do NLTK)
try:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    USE_STEMMER = True
except Exception:
    USE_STEMMER = False

warnings.filterwarnings('ignore')
DATASET = "Sports_and_Outdoors.jsonl"
SAMPLE  = 500_000   # aumentado para garantir produtos com >= 10 reviews
TOP_K   = 10

# Stopwords embutidas (não depende de download)
STOP = {
    'i','me','my','myself','we','our','ours','you','your','he','him','his',
    'she','her','it','its','they','them','what','which','who','this','that',
    'these','those','am','is','are','was','were','be','been','being','have',
    'has','had','do','does','did','will','would','could','should','may',
    'might','a','an','the','and','but','if','or','because','as','of','at',
    'by','for','with','about','to','from','in','out','on','off','not',
    'no','so','than','very','just','more','also','then','when','where',
    'there','here','was','been','get','got','one','two','three','can',
    'now','like','would','really','great','good','love','nice','use',
    'used','using','buy','bought','product','item','review','stars'
}


## 1. Carregamento

In [ ]:
records = []
with open(DATASET, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= SAMPLE: break
        try: records.append(json.loads(line))
        except: pass

df = pd.DataFrame(records)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
df['rating']    = pd.to_numeric(df['rating'], errors='coerce')
df['text']      = df['text'].fillna('')
df['title']     = df['title'].fillna('')
print(f"Registros brutos: {len(df):,}")


## 2. Pré-processamento Melhorado

In [ ]:
# Filtros mais rigorosos
prod_counts = df['asin'].value_counts()
df = df[df['asin'].isin(prod_counts[prod_counts >= 10].index)]
user_counts = df['user_id'].value_counts()
df = df[df['user_id'].isin(user_counts[user_counts >= 5].index)]
df = df[df['text'].str.split().str.len() >= 5]
df = df.drop_duplicates(subset=['user_id','asin'], keep='last')
print(f"Apos filtros: {len(df):,} | Produtos: {df['asin'].nunique():,} | Usuarios: {df['user_id'].nunique():,}")


In [ ]:
def clean(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>|http\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [t for t in text.split() if t not in STOP and len(t) > 2]
    if USE_STEMMER:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

print("Limpando textos...")
df['clean'] = (df['title'] + ' ' + df['text']).apply(clean)
print("Concluido.")


## 3. Modelo Ajustado

In [ ]:
product_profiles = (df.groupby('asin')
                      .agg(profile=('clean', lambda x: ' '.join(x)),
                           avg_rating=('rating','mean'),
                           n_reviews=('asin','count'))
                      .reset_index())

tfidf_v2 = TfidfVectorizer(
    max_features=15_000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True
)
matrix_v2 = tfidf_v2.fit_transform(product_profiles['profile'])

asin_to_idx = {a: i for i, a in enumerate(product_profiles['asin'])}
idx_to_asin = {i: a for a, i in asin_to_idx.items()}

print(f"Matriz TF-IDF v2: {matrix_v2.shape}")


In [ ]:
def recommend_v2(user_id, df_train, top_k=TOP_K):
    user_data = df_train[df_train['user_id'] == user_id]
    positive  = user_data[user_data['rating'] >= 4]
    if len(positive) == 0:
        positive = user_data
    consumed = set(user_data['asin'])
    valid = [a for a in positive['asin'] if a in asin_to_idx]
    if not valid:
        return pd.DataFrame()

    indices = [asin_to_idx[a] for a in valid]
    weights = positive[positive['asin'].isin(valid)].set_index('asin').loc[valid, 'rating'].values.astype(float)
    weights = weights / weights.sum()

    vectors = matrix_v2[indices]
    # FIX: np.asarray() converte np.matrix -> ndarray (exigido pelo cosine_similarity)
    profile = np.asarray(vectors.multiply(weights[:, np.newaxis]).sum(axis=0))
    sims = cosine_similarity(profile, matrix_v2).flatten()
    for a in consumed:
        if a in asin_to_idx:
            sims[asin_to_idx[a]] = -1

    top_idx = sims.argsort()[::-1][:top_k]
    return product_profiles[product_profiles['asin'].isin([idx_to_asin[i] for i in top_idx])].copy()

# Demonstração
sample_user = df['user_id'].value_counts().index[0]
recs = recommend_v2(sample_user, df)
print(f"Recomendacoes para {sample_user}:")
print(recs[['asin','avg_rating','n_reviews']].to_string(index=False))


## 4. Avaliação do Modelo Ajustado

In [ ]:
def temporal_split(df):
    df = df.sort_values(['user_id','timestamp'])
    train, test = [], []
    for uid, grp in df.groupby('user_id'):
        n_test = max(1, int(len(grp) * 0.2))
        if len(grp) <= n_test:
            train.append(grp)
        else:
            train.append(grp.iloc[:-n_test])
            test.append(grp.iloc[-n_test:])
    return pd.concat(train), pd.concat(test) if test else pd.DataFrame()

def precision_recall_at_k(df_train, df_test, rec_fn, k=TOP_K, max_users=300):
    relevant_map = (df_test[df_test['rating'] >= 4]
                    .groupby('user_id')['asin'].apply(set).to_dict())
    users = [u for u in list(relevant_map)[:max_users] if u in df_train['user_id'].values]
    P, R, H = [], [], []
    for u in users:
        recs = rec_fn(u, df_train, top_k=k)
        if recs.empty: continue
        rec_set = set(recs['asin'])
        rel_set = relevant_map[u]
        hits = rec_set & rel_set
        P.append(len(hits) / k)
        R.append(len(hits) / len(rel_set) if rel_set else 0)
        H.append(1 if hits else 0)
    return {'n_users': len(P),
            f'Precision@{k}': np.mean(P),
            f'Recall@{k}':    np.mean(R),
            f'HitRate@{k}':   np.mean(H)}

df_train, df_test = temporal_split(df)
print(f"Avaliando modelo v2 (K={TOP_K})...")
v2_metrics = precision_recall_at_k(df_train, df_test, recommend_v2)

print()
print("=== RESULTADOS MODELO AJUSTADO ===")
for k, v in v2_metrics.items():
    val = f"{v:,}" if isinstance(v, int) else f"{v:.4f}"
    print(f"  {k:<20}: {val}")

with open("v2_metrics.json", "w") as f:
    json.dump(v2_metrics, f)
print("\nMetricas salvas em v2_metrics.json")
